# Physics_Problems_Text_Only

In [10]:
import pandas as pd
import json

input_file = r"D:\Exact 2026\data\raw data\Physics_Problems_Text_Only.csv"
output_file = "physics_train.jsonl"

# CHANGED: system prompt bỏ yêu cầu JSON để model học reasoning tự nhiên
SYSTEM_PROMPT = "You are a physics reasoning assistant. Solve step by step and give a final answer clearly."

df = pd.read_csv(input_file, encoding="utf-8")

with open(output_file, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():

        question = str(row["question"]).strip()
        answer = str(row["answer"]).strip()
        unit = str(row["unit"]).strip() if pd.notna(row["unit"]) else ""

        if not question or question == "nan":
            continue

        # CHANGED: giữ nguyên cot dạng text, không split để tránh mất structure reasoning
        cot = str(row["cot"]).strip() if pd.notna(row["cot"]) else ""

        # CHANGED: assistant output chuyển sang text format (không JSON nữa)
        assistant_content = f"""Reasoning:
{cot}

Final Answer:
{answer} {unit}
"""

        sample = {
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": f"Solve this physics problem:\n\n{question}"
                },
                {
                    "role": "assistant",
                    "content": assistant_content
                }
            ]
        }

        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Done!")

Done!


# Logic_Based_Educational_Queries

In [11]:
import json

input_file = r"D:\Exact 2026\data\raw data\Logic_Based_Educational_Queries.json"
output_file = "logic_train.jsonl"

# CHANGED: bỏ ép JSON output, chuyển sang reasoning format tự nhiên
SYSTEM_PROMPT = "You are a logic reasoning assistant. Solve step by step and give a clear final answer."

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(output_file, "w", encoding="utf-8") as f_out:

    for item in data:

        premises_text = "\n".join([f"- {p}" for p in item["premises-NL"]])

        for i in range(len(item["questions"])):

            user_content = f"""Solve this logic problem:

Premises:
{premises_text}

Question:
{item["questions"][i]}"""

            # CHANGED: bỏ split explanation, giữ nguyên reasoning tự nhiên
            explanation = item["explanation"][i]

            # CHANGED: assistant trả text thay vì JSON
            assistant_content = f"""Answer:
{item["answers"][i]}

Reasoning:
{explanation}
"""

            sample = {
                "messages": [
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": user_content
                    },
                    {
                        "role": "assistant",
                        "content": assistant_content
                    }
                ]
            }

            f_out.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Done!")

Done!


# Z3

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("yale-nlp/FOLIO",
                  token="")

In [12]:
def convert_folio(example):
    premises_raw = example.get("premises", [])
    conclusion = example.get("conclusion", "")

    if isinstance(premises_raw, list):
        premises_text = "\n".join(premises_raw)
    else:
        premises_text = str(premises_raw)

    # CHANGED: bỏ JSON output, chuyển sang reasoning text format
    label = "Yes" if example["label"] == "entailment" else "No"

    reasoning_text = f"""Given the premises, we evaluate whether the conclusion logically follows.

If the premises entail the conclusion, the answer is Yes; otherwise No.

Answer: {label}
"""

    return {
        "messages": [
            {
                "role": "system",
                "content": "You are a logical reasoning assistant. Solve step by step and give a clear final answer."
            },
            {
                "role": "user",
                "content": f"Premises:\n{premises_text}\n\nConclusion:\n{conclusion}"
            },
            {
                "role": "assistant",
                "content": reasoning_text
            }
        ]
    }


output_file = "folio_train.jsonl"

with open(output_file, "w", encoding="utf-8") as f:

    for example in ds["train"]:
        sample = convert_folio(example)
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

    for example in ds["validation"]:
        sample = convert_folio(example)
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Done! Saved to:", output_file)

Done! Saved to: folio_train.jsonl
